In [1]:
# --- Colab bootstrap -------------------------------------------------------
import os, sys
if not (os.path.isdir('../thermo') or os.path.isdir('thermo')):
    !rm -rf /tmp/cbet6e
    !git clone -q --depth 1 https://github.com/emfurst/cbet6e.git /tmp/cbet6e
    !cp -r /tmp/cbet6e/code/thermo /tmp/cbet6e/code/data .
    sys.path.insert(0, '.')
# ---------------------------------------------------------------------------

# One chemical, one measurement, five compartments (Illustrations 12.1-3, 12.5-1 and 12.5-3)

Benzo[a]pyrene is a combustion product and a carcinogen. Three illustrations of this
chapter begin from a single measured property of it -- a solubility,
$x_{\rm BP} = 3.37\times10^{-10}$ in water at 25 °C -- and end with concentrations in five
environmental compartments:

1. **Illustration 12.1-3** converts the solubility to an activity coefficient. Because
   benzo[a]pyrene is a *solid*, $\gamma\ne1/x^{\rm sat}$: the saturated solution is in
   equilibrium with the pure solid, so the fugacity ratio of Eq. 12.1-6 has to be divided
   out first. The result is $\gamma^\infty = 3.76\times10^8$. The Comment on the
   illustration uses that number to make the point that activity coefficients are not
   always small corrections.
2. **Illustration 12.5-1** converts that activity coefficient to an air-water partition
   coefficient through Eq. 12.5-8.
3. **Illustration 12.5-3** converts that partition coefficient, plus a measured
   $K_{\rm OW}$, to concentrations in air, soil, sediment and fish, and compares them with
   field data spanning eight orders of magnitude.

The model makes one substitution, and makes it three times: each compartment is treated as
*the fraction of itself that behaves like octanol* -- lipid in fish, organic carbon in soil
and sediment. Equations 12.5-9 to 12.5-11 are $K_{\rm OW}$ times that fraction, with a
factor of 0.4 for organic carbon that Sec. 12.5 identifies as empirical.

Every concentration printed in Illustration 12.5-3 reproduces here to the digits given.
The comparison with the field data, in the last cell, is where the calculation and the
measurements part: air -- the one compartment reached through $\gamma^\infty$ and a vapor
pressure -- falls inside the reported range, while the three reached through the octanol
substitution are high, soil by 3.4 and fish by 11. Give lipid the same empirical 0.4 that
Sec. 12.5 gives organic carbon and fish comes to 4.4, within a factor of 1.3 of soil. That
puts the disagreement in the substitution rather than in Eqs. 12.5-5 to 12.5-8.

The book has printed two different values of $K_{\rm AW}$ for the same quantity. Illustration
12.5-1 computed $5.844\times10^{-5}$; Illustration 12.5-3 quoted and used
$5.884\times10^{-5}$; and neither is what Eq. 12.5-8 returns. All three numbers are
accounted for below. The equation gives $5.813\times10^{-5}$. The original worksheet
included a spurious factor of the total pressure, 1.013 bar, which gives
$5.884\times10^{-5}$ and is what Illustration 12.5-3 used. And $5.844$ is $5.884$ with two
digits transposed. The 6e prints the equation's own value, $5.813\times10^{-5}$, in both
places, which moves one number in Illustration 12.5-3's results table: the air
concentration, from 1.66 to 1.64 ng/m$^3$. The reported range of 1.3 to 7.1 contains both.

SIS is Stanley I. Sandler, *Chemical, Biochemical, and Engineering Thermodynamics*.

Eric Furst
August 2026

In [2]:

import sys; sys.path.insert(0, "..")
import numpy as np

from thermo import sle
from thermo.partition import (air_water_partition, compartment_partition,
                              compartment_concentrations, COMPARTMENTS,
                              EQ_12_5_8_CONSTANT)

T = 298.15

# --- Illustration 12.1-3 ---------------------------------------------------
TM_BP    = 178.1 + 273.15          # melting point, K
DH_FUS   = 15100.0                 # J/mol
X_SAT    = 3.37e-10                # mole fraction in water at 25 C
P_VAP_PA = 2.13e-5                 # Pa, the extrapolated liquid vapor pressure
LOG_KOW  = 6.04                    # measured

print("  Illustration 12.1-3: an activity coefficient from a solubility")
ratio = float(np.exp(sle.ln_x_gamma(T, TM_BP, DH_FUS)))
gamma = float(sle.activity_coefficient(X_SAT, T, TM_BP, DH_FUS))
print(f"    f^S / f^L         = {ratio:.4f}")
print(f"    gamma^inf         = {gamma:.4g}      SIS 3.76e8")
print(f"    1/x_sat would be  = {1 / X_SAT:.4g}"
      f"   -- high by a factor of {1 / ratio:.2f}")
print(f"\n    That factor is why a melting point appears in what looks like a")
print(f"    solubility calculation. Benzo[a]pyrene is a solid at 25 C, so a saturated")
print(f"    aqueous solution is in equilibrium with the pure solid, not with a pure")
print(f"    liquid -- and the ratio of those two fugacities is Eq. 12.1-6's exponential.")
print(f"    Taking gamma = 1/x_sat overpredicts by a factor of {1 / ratio:.1f}.")

  Illustration 12.1-3: an activity coefficient from a solubility
    f^S / f^L         = 0.1266
    gamma^inf         = 3.757e+08      SIS 3.76e8
    1/x_sat would be  = 2.967e+09   -- high by a factor of 7.90

    That factor is why a melting point appears in what looks like a
    solubility calculation. Benzo[a]pyrene is a solid at 25 C, so a saturated
    aqueous solution is in equilibrium with the pure solid, not with a pure
    liquid -- and the ratio of those two fugacities is Eq. 12.1-6's exponential.
    Taking gamma = 1/x_sat overpredicts by a factor of 7.9.


## Illustration 12.5-1, and the constant in Eq. 12.5-8

Equation 12.5-8 is $K_{\rm AW} = 0.2164\,\gamma^\infty P^{\rm vap}/T$ for $P^{\rm vap}$
in bar. The 0.2164 is not fitted; it is three unit conversions multiplied together, and
one of the three is the factor by which the two illustrations disagree:

$$\underbrace{1.218\times10^4}_{\text{Eq. 12.5-6, ideal gas}}
  \Big/ \underbrace{5.556\times10^4}_{\text{Eq. 12.5-7},\;10^6/18}
  \Big/ \underbrace{1.013}_{P\ \text{in Eq. 12.5-5}} = 0.2164$$

The molecular weight cancels between Eqs. 12.5-6 and 12.5-7, which is why it does not
appear. The atmospheric pressure comes in because Eq. 12.5-5 is
$y_iP = x_i\gamma_i^\infty P_i^{\rm vap}$, so $y_i/x_i$ contains a $1/P$.

In [3]:

print("  Where 0.2164 comes from")
print(f"    1.218e4 / 5.556e4        = {1.218e4 / 5.556e4:.5f}   SIS's own 0.2192")
print(f"    divided by P = 1.013 bar = {EQ_12_5_8_CONSTANT:.5f}   SIS's own 0.2164")

print("\n  Illustration 12.5-1, three ways of arriving at K_AW")
K_EQ = float(air_water_partition(3.76e8, P_VAP_PA * 1e-5, T))
K_NOP = float(air_water_partition(3.76e8, P_VAP_PA * 1e-5, T,
                                  constant=1.218e4 / 5.556e4))
K_EXACT = float(air_water_partition(gamma, P_VAP_PA * 1e-5, T))
print(f"    Eq. 12.5-8, gamma = 3.76e8 as printed   K_AW = {K_EQ:.4e}")
print(f"    the same with the unrounded gamma       K_AW = {K_EXACT:.4e}")
print(f"    the same but without the 1/1.013        K_AW = {K_NOP:.4e}")
print(f"\n    SIS, Illustration 12.5-1               K_AW = 5.844e-05")
print(f"    SIS, Illustration 12.5-3 (quoted and used)   5.884e-05")
K_NOP_EXACT = float(air_water_partition(gamma, P_VAP_PA * 1e-5, T,
                                        constant=1.218e4 / 5.556e4))
print(f"    the same, unrounded gamma and no 1/1.013 K_AW = {K_NOP_EXACT:.4e}")
print(f"\n  All three printed numbers are accounted for:")
print(f"    * Eq. 12.5-8 as printed returns {K_EQ:.4e}. Its constant 0.2164 agrees with")
print(f"      V_W/R = 0.2167 to within 0.2 %, with T in K and P^vap in bar.")
print(f"    * dropping the 1/1.013, with the unrounded gamma = {gamma:.4e} from")
print(f"      Illustration 12.1-3, gives {K_NOP_EXACT:.4e}, which is Illustration")
print(f"      12.5-3's printed 5.884e-5 to four figures. That identifies the route the")
print(f"      original worksheet took.")
print(f"    * 5.844e-5 in Illustration 12.5-1 is a digit transposition of 5.884, and is")
print(f"      not the output of any route.")
print(f"\n  The 6e prints {K_EQ:.4e} in both illustrations. Only the air concentration in")
print(f"  Illustration 12.5-3 moves, by {abs(K_EQ / 5.884e-5 - 1) * 100:.1f} %.")

  Where 0.2164 comes from
    1.218e4 / 5.556e4        = 0.21922   SIS's own 0.2192
    divided by P = 1.013 bar = 0.21641   SIS's own 0.2164

  Illustration 12.5-1, three ways of arriving at K_AW
    Eq. 12.5-8, gamma = 3.76e8 as printed   K_AW = 5.8131e-05
    the same with the unrounded gamma       K_AW = 5.8084e-05
    the same but without the 1/1.013        K_AW = 5.8887e-05

    SIS, Illustration 12.5-1               K_AW = 5.844e-05
    SIS, Illustration 12.5-3 (quoted and used)   5.884e-05
    the same, unrounded gamma and no 1/1.013 K_AW = 5.8839e-05

  All three printed numbers are accounted for:
    * Eq. 12.5-8 as printed returns 5.8131e-05. Its constant 0.2164 agrees with
      V_W/R = 0.2167 to within 0.2 %, with T in K and P^vap in bar.
    * dropping the 1/1.013, with the unrounded gamma = 3.7570e+08 from
      Illustration 12.1-3, gives 5.8839e-05, which is Illustration
      12.5-3's printed 5.884e-5 to four figures. That identifies the route the
      original worksh

## Illustration 12.5-3: five compartments, eight orders of magnitude

Given the concentration in water -- $2.82\times10^4$ ng/m$^3$ in southern Ontario -- every
other compartment follows from a partition coefficient, and each one is a single
multiplication. The errors available here are unit errors. Each compartment answer is
wanted per unit *volume* while the partition coefficients are per unit *mass*, and the two
differ by the compartment density: 1500 kg/m$^3$ for soil, 1420 for sediment, about 1000
for biota. Interchanging them changes the soil answer by a factor of 1.5, and reporting
ng/g where ppm by weight is wanted changes it by a factor of $10^3$. Neither mistake makes
the answer look wrong.

`compartment_concentrations` therefore returns both bases, named, rather than one bare
array.

In [4]:

C_WATER = 2.82e4                   # ng/m^3, reported for southern Ontario
KOW = 10.0 ** LOG_KOW
# The 6e prints Eq. 12.5-8's own answer in both illustrations. Earlier printings gave
# 5.844e-5 in Illustration 12.5-1 and used 5.884e-5 in Illustration 12.5-3, and
# neither is what the equation returns -- see the cell above and the note below.
K_AW = K_EQ                        # 5.813e-5, from Eq. 12.5-8 as printed
K_AW_ALT = 5.884e-5                 # what Illustration 12.5-3 used

print(f"  K_OW = 10^{LOG_KOW} = {KOW:.4g}    SIS 1.096e6")
print("\n  The partition coefficients, Eqs. 12.5-9 to 12.5-11")
SIS_K = {"biota": 5.48e4, "soil": 8768.0, "sediment": 21920.0}
print(f"    {'compartment':<11} {'w':>6} {'factor':>7} {'K':>12} {'SIS':>11}")
for name in ("biota", "soil", "sediment"):
    c = COMPARTMENTS[name]
    K = float(compartment_partition(KOW, name))
    print(f"    {name:<11} {c['w']:6.2f} {c['factor']:7.1f} {K:12.4g}"
          f" {SIS_K[name]:11.4g}")
print(f"    (SIS's biota value is 0.05 x K_OW and is not printed as a K, only used.)")

res = compartment_concentrations(C_WATER, KOW, K_AW)
res_alt = compartment_concentrations(C_WATER, KOW, K_AW_ALT)
SIS_CALC = {"air": 1.66, "soil": 3.71e8, "sediment": 8.78e8, "biota": 1.55e9}
# Only the air concentration moves when K_AW changes: soil, sediment and biota
# come from K_OW through Eqs. 12.5-9 to 12.5-11 and never see the air-water
# coefficient, which is why the 6e correction to K_AW reaches one number in the
# printed table.
SIS_6E = {"air": 1.64}
SIS_REPORTED = {"air": "1.3 to 7.1", "soil": "1.1e8",
                "sediment": "0.8e8 to 3e8", "biota": "1.4e8"}
print(f"\n  Concentrations, ng/m^3")
print(f"    {'compartment':<11} {'recomputed':>12} {'SIS calc':>11} {'ratio':>7}"
      f"   reported (Mackay and Paterson)")
print(f"    {'water':<11} {C_WATER:12.4g} {C_WATER:11.4g} {1.0:7.3f}"
      f"   2.82e4 (the input)")
for name in ("air", "soil", "sediment", "biota"):
    v = float(res[name]["per_volume"])
    print(f"    {name:<11} {v:12.4g} {SIS_CALC[name]:11.4g}"
          f" {v / SIS_CALC[name]:7.3f}   {SIS_REPORTED[name]}")

print(f"\n  What the K_AW correction moves, and what it does not")
for name in ("air", "soil", "sediment", "biota"):
    a = float(res[name]["per_volume"]); b = float(res_alt[name]["per_volume"])
    flag = "  <- the only one that moves" if abs(a / b - 1) > 1e-9 else ""
    print(f"    {name:<11} K_AW = 5.813e-5: {a:11.4g}   5.884e-5: {b:11.4g}"
          f"  {(a / b - 1) * 100:+5.2f} %{flag}")
print(f"    so the 6e prints {SIS_6E['air']} ng/m^3 for air where the earlier text printed"
      f" {SIS_CALC['air']}, and the")
print(f"    reported range {SIS_REPORTED['air']} ng/m^3 contains both.")

print(f"\n  And the ppm-by-weight figures the illustration also prints")
# per_mass comes back in ng per gram of compartment, which is ppb by weight; ppm is
# that over 1000. Getting this step wrong is a factor of 1e3 and leaves the answer in
# the range a reader would accept, which is why `compartment_concentrations` names the
# basis of both columns instead of returning one bare array.
for name, sis in (("soil", 0.247), ("sediment", 0.618), ("biota", 1.55)):
    pm = float(res[name]["per_mass"]) / 1e3     # ng/g = ppb -> ppm by weight
    print(f"    {name:<11} {pm:8.4f} ppm    SIS {sis}")

  K_OW = 10^6.04 = 1.096e+06    SIS 1.096e6

  The partition coefficients, Eqs. 12.5-9 to 12.5-11
    compartment      w  factor            K         SIS
    biota         0.05     1.0    5.482e+04    5.48e+04
    soil          0.02     0.4         8772        8768
    sediment      0.05     0.4    2.193e+04   2.192e+04
    (SIS's biota value is 0.05 x K_OW and is not printed as a K, only used.)

  Concentrations, ng/m^3
    compartment   recomputed    SIS calc   ratio   reported (Mackay and Paterson)
    water           2.82e+04    2.82e+04   1.000   2.82e4 (the input)
    air                1.639        1.66   0.988   1.3 to 7.1
    soil            3.71e+08    3.71e+08   1.000   1.1e8
    sediment       8.781e+08    8.78e+08   1.000   0.8e8 to 3e8
    biota          1.546e+09    1.55e+09   0.997   1.4e8

  What the K_AW correction moves, and what it does not
    air         K_AW = 5.813e-5:       1.639   5.884e-5:       1.659  -1.20 %  <- the only one that moves
    soil        K_AW 

In [5]:

# Reported ranges from Mackay and Paterson, as Illustration 12.5-3 gives them. Where a
# single value is reported both ends are set equal, so the ratio column below never
# hides a midpoint the reader cannot see.
REPORTED = {"air": (1.3, 7.1), "soil": (1.1e8, 1.1e8),
            "sediment": (0.8e8, 3.0e8), "biota": (1.4e8, 1.4e8)}

print("  Comparison with the field data")
calc_span = SIS_CALC["biota"] / SIS_CALC["air"]
rep_span = REPORTED["biota"][1] / REPORTED["air"][0]
print(f"    calculated, air to fish: {np.log10(calc_span):.1f} orders of magnitude,"
      f" {SIS_CALC['air']:.2g} to {SIS_CALC['biota']:.2g} ng/m^3")
print(f"    reported,   air to fish: {np.log10(rep_span):.1f} orders of magnitude,"
      f" {REPORTED['air'][0]:.2g} to {REPORTED['biota'][1]:.2g} ng/m^3")
print(f"\n    One measured solubility, one measured K_OW, and no fitted parameter beyond")
print(f"    the empirical 0.4 put every compartment within a factor of 11 of measurement")
print(f"    across nine orders of magnitude. The disagreement is not uniform:")

print(f"\n    {'compartment':<11} {'calculated':>11}   {'reported':>17}   calc/reported")
for name in ("air", "soil", "sediment", "biota"):
    calc = float(res[name]["per_volume"])
    lo, hi = REPORTED[name]
    rng = f"{lo:.2g}" if lo == hi else f"{lo:.2g} to {hi:.2g}"
    rat = f"{calc / hi:.2f}" if lo == hi else f"{calc / hi:.2f} to {calc / lo:.2f}"
    print(f"    {name:<11} {calc:11.4g}   {rng:>17}   {rat}")
print(f"    (Where a range is reported, the ratio is given against both of its ends.)")

r_soil = float(res["soil"]["per_volume"]) / REPORTED["soil"][0]
r_biota = float(res["biota"]["per_volume"]) / REPORTED["biota"][0]
r_lipid_04 = r_biota * COMPARTMENTS["soil"]["factor"]
spread = max(r_soil, r_lipid_04) / min(r_soil, r_lipid_04)

print(f"\n  Where the disagreement sits")
print(f"    Air is the only compartment reached through the thermodynamics of this")
print(f"    chapter -- gamma^inf from a solubility, then Eq. 12.5-8 -- and it is the only")
print(f"    one inside its reported range. Soil, sediment and fish are reached instead")
print(f"    through a measured K_OW and an assumption about what fraction of each")
print(f"    compartment behaves like octanol, and all three come out high.")
print(f"\n    soil, Eq. 12.5-10   high by {r_soil:5.1f}   w = 0.02, times the empirical 0.4")
print(f"    fish, Eq. 12.5-11   high by {r_biota:5.1f}   w = 0.05, with no such factor")
print(f"\n    The two equations differ in one thing: the organic-carbon coefficient")
print(f"    includes the 0.4 and the lipid coefficient does not. Applying the same 0.4 to")
print(f"    lipid gives {r_lipid_04:.1f}, within a factor of {spread:.1f} of soil, and the reported")
print(f"    sediment range brackets both. The three K_OW compartments are therefore")
print(f"    consistent with one another under a single missing correction to the octanol")
print(f"    substitution. That locates the error in the substitution and not in")
print(f"    Eqs. 12.5-5 to 12.5-8, and question 1 below is the test of it: fit the")
print(f"    fraction and see whether one value serves all three compartments.")

  Comparison with the field data
    calculated, air to fish: 9.0 orders of magnitude, 1.7 to 1.6e+09 ng/m^3
    reported,   air to fish: 8.0 orders of magnitude, 1.3 to 1.4e+08 ng/m^3

    One measured solubility, one measured K_OW, and no fitted parameter beyond
    the empirical 0.4 put every compartment within a factor of 11 of measurement
    across nine orders of magnitude. The disagreement is not uniform:

    compartment  calculated            reported   calc/reported
    air               1.639          1.3 to 7.1   0.23 to 1.26
    soil           3.71e+08             1.1e+08   3.37
    sediment      8.781e+08      8e+07 to 3e+08   2.93 to 10.98
    biota         1.546e+09             1.4e+08   11.04
    (Where a range is reported, the ratio is given against both of its ends.)

  Where the disagreement sits
    Air is the only compartment reached through the thermodynamics of this
    chapter -- gamma^inf from a solubility, then Eq. 12.5-8 -- and it is the only
    one inside 

## Your turn

1. Section 12.5 takes the organic-carbon-water partition coefficient as 40 % of
   $K_{\rm OW}$, a value it describes as empirically found. Fit that fraction instead,
   from the reported soil and sediment concentrations, and then from the reported fish
   concentration as well. Does one number serve all three, and how far is it from 0.4?
2. Fish is the compartment furthest from measurement, and its coefficient rests on lipid
   being octanol at a weight fraction of 0.05. What lipid fraction reproduces the reported
   $1.4\times10^8$ ng/m$^3$? Is that a plausible fish?
3. Problem 12.5-2 gives four insecticides with their water solubilities and
   $\log_{10}K_{\rm OW}$, and asks for the concentration in a fish in a saturated tank.
   Do all four, and notice that solubility and $K_{\rm OW}$ run in opposite directions --
   which of the two decides the answer?
4. Problem 12.5-3 is a closed terrarium: 10 m$^3$ total, contaminated with 10 mg of
   benzene. That is a *mass balance* problem, not a partition-coefficient problem -- the
   total is fixed and shared out. Set it up and solve for all four compartments, then do
   it again for DDT and explain the difference in one sentence.
5. Illustration 12.1-3's route needs a melting point and a heat of fusion because the
   solute is a solid. What would change if benzo[a]pyrene were a liquid at 25 °C, and
   which of this notebook's numbers would move?
6. The vapor pressure used in Illustration 12.5-1 is described as *extrapolated* -- it is
   the pressure of a liquid that does not exist at 25 °C, the same hypothetical phase as
   in Illustration 12.1-1. Trace how a 20 % error in that extrapolation propagates to the
   air concentration, and compare that sensitivity with the 0.4.